# 04 Full extraction: every record checked against its PDF, the extraction tables, the decisive values, gate G3

Design log.
- Plan v2 B3 says this notebook validates every record file, builds the CSV tables, and lists the decisive values for direct checking. The second-pass sample and its agreement figures were split out into notebook 04_second_pass, and the full re-code of the seven low-agreement fields into notebook 04r. Both are committed, so this notebook starts from the re-coded records.
- Stage 1 stops unless 04r is committed and says the amended rules hold in every record.
- Stage 3 checks all 124 records in one run, not batch by batch:
  - allowed values and required fields (the frozen schema v1);
  - a page for every paper-sourced field, and every page within its PDF;
  - read status F for every report;
  - who measured every evaluation, and whether it is the original measurement or a later one;
  - every printed value, chart value and anchor found on its page. Pages are read with plain and column-preserving text from pypdf; a value still not found is looked for in a third reading with pypdfium2;
  - the PDF's SHA-256 against `reports/defences.csv`;
  - the amended coding rules (Amendment 2).
  Nothing is written to `reports/extraction/` unless every check passes.
- Stage 4 writes the tables to `reports/extraction/`. They go there, not straight into `reports/`, because `reports/defences.csv` is already the PDF manifest. There is one row per record, and ids are global (`<report_id>:<local_id>`). Two columns are worked out by code: original or later measurement, and the comparison grade.
- Stage 5 lists the decisive values. Plan A6 defines them as every number that appears in the abstract, in a figure or in a grade-A comparison. The review's abstract and figures do not exist yet, so the list here holds every printed value in a grade-A comparison. For each value it records:
  - its address (page, table, arm, benchmark, metric, unit);
  - which text reading found it on the page;
  - how many times that number stands on the page.
  A value that occurs once on its page is pinned to that place by code. A value that occurs more than once, or only in the third reading, can only be confirmed by reading its cell. Those cells are read against the page for every value the findings of notebook 05 use, before the figures are drawn.
- Stage 6 applies gate G3:
  - full-text rows exist in the cells of the map;
  - layouts are stated for most comparisons;
  - agreement above the thresholds, or the low-agreement fields re-coded in full. Here the fields were re-coded in 04r.
  It also writes `decision_04.txt` with the number of full-text rows in each cell of the map.
- CPU only; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. Gate: the re-code of notebook 04r is committed and the amended rules hold

In [ ]:
NEED = ["reports/decision_04sp.txt", "reports/agreement_04.csv", "reports/decision_04r.txt", "reports/recode_04r.csv",
        "reports/recode_04r_changes.csv", "reports/second_pass_resolution_04r.csv"]
missing = [p for p in NEED if not Path(p).exists()]
untracked = bool([p for p in NEED if Path(p).exists()]) and subprocess.run(
    ["git", "ls-files", "--error-unmatch", *[p for p in NEED if Path(p).exists()]], capture_output=True, text=True).returncode != 0
dirty = subprocess.run(["git", "status", "--porcelain", "--", *NEED, "reports/records", "schema"], capture_output=True, text=True).stdout.strip()
if missing or untracked or dirty:
    raise SystemExit(f"STOP: run and commit notebook 04r first (missing: {missing}; not committed: {untracked}; changed since commit: {dirty[:300]})")
d04r = Path("reports/decision_04r.txt").read_text().splitlines()
if "AMENDED RULES HOLD IN EVERY RECORD" not in d04r[0]:
    raise SystemExit("STOP: decision_04r.txt does not report the amended rules holding in every record")
R_COMMIT = subprocess.run(["git", "log", "-1", "--format=%h", "--", "reports/decision_04r.txt"], capture_output=True, text=True).stdout.strip()
print("04r committed in", R_COMMIT, "|", d04r[0])

## Stage 2. The extraction module (decisive values and the map added) and its tests

In [ ]:
%%writefile src/extract.py
"""Record files to CSV tables, the second-pass sample, value cards and agreement statistics (plan v2, B4).

Notebook 04 writes this module. Nothing here calls a model or the network."""
from __future__ import annotations

import hashlib
import json
import math
import re
from collections import Counter, defaultdict

SEED = 2026
SHARE = 0.2
CARDS_PER_REPORT = 6
QUANTITIES = ["R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe"]
DEF_CATEGORICAL = ["origin", "isolation", "signal_dependence", "trains_instruction_pair", "trains_carry_through"]
DEF_MULTI = ["axis1", "axis2"]
EXACT_MIN, KAPPA_MIN, JACCARD_MIN = 0.8, 0.6, 0.8
MISSING = "(missing)"


def rank_key(seed, *parts) -> str:
    return hashlib.sha256("|".join([str(seed), *map(str, parts)]).encode()).hexdigest()


def second_pass_sample(included_slugs, seed: int = SEED, share: float = SHARE) -> list[str]:
    """The seeded sample: every included study ranked by a hash of the seed and its slug, the first
    ceil(share * N) kept. Removing a study from the frame does not reorder the others."""
    slugs = sorted(set(included_slugs))
    n = math.ceil(share * len(slugs))
    return sorted(slugs, key=lambda s: rank_key(seed, s))[:n]


def value_cards(rec: dict, k: int = CARDS_PER_REPORT, seed: int = SEED) -> list[dict]:
    """Up to k printed results per report, chosen by hash rank, given by their address only (never the value)."""
    rid = rec["report"]["id"]
    evs = {e["id"]: e for e in rec.get("evaluations") or []}
    dfs = {d["id"]: d for d in rec.get("defences") or []}
    printed = [r for r in rec.get("results") or [] if r.get("value_source") != "figure_only" and r.get("value") not in (None, "")]
    printed.sort(key=lambda r: rank_key(seed, rid, r["id"]))
    cards = []
    for r in printed[:k]:
        e = evs[r["evaluation_id"]]
        d = dfs.get(e["defence_id"])
        arm = f"{d['name']} on {d['base_checkpoint']}" if d else f"no defence ({str(e['defence_id'])[5:]}, {e.get('baseline_kind', '')})"
        extra = "; ".join(f"{f}: {e[f]}" for f in ("split", "run", "injection_position", "attacker_budget", "subset_of") if e.get(f))
        cards.append({"report": rid, "result": r["id"], "page": r.get("page"), "location": r.get("location", ""),
                      "arm": arm, "benchmark": e["benchmark"], "attacker": e["attacker_type"], "judge": e["judge"],
                      "metric": r["metric_as_printed"], "quantity": r["quantity"], "unit": r["unit"], "detail": extra})
    return cards


def _cell(v):
    if v is None:
        return ""
    if isinstance(v, list):
        return ";".join(str(x) for x in v)
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True, ensure_ascii=False)
    return v


def build_tables(records: dict, schema: dict, grade) -> dict:
    """The five extraction tables (defences, evaluations, results, comparisons, claims) and a report index, with
    ids made global as <report_id>:<local_id>. Derived columns: the defence each result belongs to, whether an
    evaluation is the original or a later measurement, and the grade of each comparison."""
    k = schema["kinds"]
    out = {name: [] for name in ("reports", "defences", "evaluations", "results", "comparisons", "claims")}
    for rid in sorted(records):
        rec = records[rid]
        rep, cov = rec["report"], rec.get("coverage") or {}
        dfs = {d["id"]: d for d in rec.get("defences") or []}
        evs = {e["id"]: e for e in rec.get("evaluations") or []}
        out["reports"].append({**{f: _cell(rep.get(f)) for f in k["report"]["fields"]},
                               **{f"cov_{q}": cov.get(q, "") for q in QUANTITIES},
                               "cov_where": _cell(cov.get("where")),
                               **{f"n_{x}": len(rec.get(x) or []) for x in ("defences", "evaluations", "results", "comparisons", "claims")}})
        for d in rec.get("defences") or []:
            out["defences"].append({"report_id": rid, "defence_uid": f"{rid}:{d['id']}", **{f: _cell(d.get(f)) for f in k["defences"]["fields"]}})

        def duid(did):
            did = str(did)
            return did if did.startswith("none:") or ":" in did else f"{rid}:{did}"

        def measurement(e):
            d = dfs.get(e["defence_id"])
            if d is None:
                return "undefended" if str(e["defence_id"]).startswith("none:") else "other_record"
            if d.get("origin") == "this_report" and e.get("measured_by") == rid:
                return "original"
            if d.get("origin") == "this_report":
                return "original_reprinted"
            return "later"

        for e in rec.get("evaluations") or []:
            out["evaluations"].append({"report_id": rid, "evaluation_uid": f"{rid}:{e['id']}", "defence_uid": duid(e["defence_id"]),
                                       "measurement": measurement(e), **{f: _cell(e.get(f)) for f in k["evaluations"]["fields"]}})
        for r in rec.get("results") or []:
            e = evs.get(r["evaluation_id"], {})
            out["results"].append({"report_id": rid, "result_uid": f"{rid}:{r['id']}", "evaluation_uid": f"{rid}:{r['evaluation_id']}",
                                   "defence_uid": duid(e.get("defence_id", "")), "measurement": measurement(e) if e else "",
                                   **{f: _cell(r.get(f)) for f in k["results"]["fields"]}})
        for c in rec.get("comparisons") or []:
            out["comparisons"].append({"report_id": rid, "comparison_uid": f"{rid}:{c['id']}", "left_uid": f"{rid}:{c['left']}",
                                       "right_uid": f"{rid}:{c['right']}", "grade": grade(c),
                                       **{f: _cell(c.get(f)) for f in k["comparisons"]["fields"]}})
        for cl in rec.get("claims") or []:
            row = {"report_id": rid, "claim_uid": f"{rid}:{cl['id']}", **{f: _cell(cl.get(f)) for f in k["claims"]["fields"]}}
            row["supported_by"] = ";".join(f"{rid}:{s}" for s in cl.get("supported_by") or [])
            row["contradicted_by"] = ";".join(f"{rid}:{s}" for s in cl.get("contradicted_by") or [])
            out["claims"].append(row)
    fields = {name: (list(rows[0].keys()) if rows else []) for name, rows in out.items()}
    return {name: (rows, fields[name]) for name, rows in out.items()}


def cross_record_problems(records: dict, included_slugs, page_counts: dict | None = None) -> list[str]:
    """Checks that need the whole corpus: one record per included study, references to other reports resolve,
    every record read in full, and (when page counts are given) every page within its PDF."""
    probs = []
    by_slug = defaultdict(list)
    for rid, rec in records.items():
        by_slug[rec["report"]["slug"]].append(rid)
    inc = set(included_slugs)
    for s in sorted(inc - set(by_slug)):
        probs.append(f"included study {s} has no record")
    for s in sorted(set(by_slug) - inc):
        probs.append(f"record(s) {by_slug[s]} belong to {s}, which is not an included study")
    for s, rids in by_slug.items():
        if len(rids) > 1:
            probs.append(f"included study {s} has {len(rids)} records: {rids}")
    defences = {f"{rid}:{d['id']}" for rid, rec in records.items() for d in rec.get("defences") or []}
    for rid, rec in records.items():
        if rec["report"].get("read") != "F":
            probs.append(f"{rid}: read is {rec['report'].get('read')!r}, not F")
        for e in rec.get("evaluations") or []:
            mb = e.get("measured_by")
            if mb != rid and mb not in records:
                probs.append(f"{rid}:{e['id']}: measured_by {mb!r} is neither this report nor another record")
            sc = e.get("same_checkpoint_as")
            if sc and sc not in defences:
                probs.append(f"{rid}:{e['id']}: same_checkpoint_as {sc!r} is not a defence in the corpus")
            did = str(e.get("defence_id", ""))
            if ":" in did and not did.startswith("none:") and did not in defences:
                probs.append(f"{rid}:{e['id']}: defence_id {did!r} is not a defence in the corpus")
        if page_counts and rid in page_counts:
            n = page_counts[rid]
            for kind in ("defences", "evaluations", "results", "comparisons", "claims"):
                for row in rec.get(kind) or []:
                    pages = [row.get("page")] + list((row.get("pages") or {}).values())
                    for p in pages:
                        if isinstance(p, int) and p > n:
                            probs.append(f"{rid}:{kind}:{row.get('id')}: page {p} beyond the PDF's {n} pages")
    return probs


def validate_second_pass(doc: dict, schema: dict, cards: list[dict], first: dict, match_line: list) -> list[str]:
    """Allowed values in a second-pass document, value answers for exactly the drawn cards, and match lines that
    point at first-pass defences of the same report."""
    k, rid, out = schema["kinds"], doc.get("report"), []
    if doc.get("role") not in k["report"]["fields"]["role"]["values"]:
        out.append(f"{rid}: role {doc.get('role')!r} not allowed")
    for q in QUANTITIES:
        if (doc.get("coverage") or {}).get(q) not in k["coverage"]["fields"][q]["values"]:
            out.append(f"{rid}: coverage {q} {(doc.get('coverage') or {}).get(q)!r} not allowed")
    for e in doc.get("defences") or []:
        for f in DEF_CATEGORICAL:
            if e.get(f) not in k["defences"]["fields"][f]["values"]:
                out.append(f"{rid}: {e.get('name')}: {f} {e.get(f)!r} not allowed")
        for f in DEF_MULTI:
            vals = e.get(f) or []
            if not vals or any(v not in k["defences"]["fields"][f]["values"] for v in vals):
                out.append(f"{rid}: {e.get('name')}: {f} {vals!r} not allowed")
        if not e.get("bases"):
            out.append(f"{rid}: {e.get('name')}: no base models listed")
    answered = [v["result"] for v in doc.get("values") or []]
    drawn = [c["result"] for c in cards]
    if sorted(answered) != sorted(drawn):
        out.append(f"{rid}: values answered {sorted(answered)} differ from the cards drawn {sorted(drawn)}")
    ids = {d["id"] for d in first.get("defences") or []}
    if len(match_line or []) != len(doc.get("defences") or []):
        out.append(f"{rid}: {len(match_line or [])} match lines for {len(doc.get('defences') or [])} defence entries")
    for ids_ in match_line or []:
        for i in ids_ or []:
            if i not in ids:
                out.append(f"{rid}: match {i!r} is not a first-pass defence")
    used = [i for ids_ in match_line or [] for i in ids_ or []]
    for i in {x for x in used if used.count(x) > 1}:
        out.append(f"{rid}: first-pass defence {i!r} matched twice")
    return out


# ---- agreement --------------------------------------------------------------------------------------------------

def cohen_kappa(pairs: list[tuple]) -> float | None:
    """Cohen's kappa over (first, second) pairs; missing values are a category. None when chance agreement is 1
    (every value in one category), where kappa is undefined and exact agreement is reported alone."""
    n = len(pairs)
    if not n:
        return None
    pairs = [(MISSING if a in (None, "") else a, MISSING if b in (None, "") else b) for a, b in pairs]
    po = sum(a == b for a, b in pairs) / n
    ca, cb = Counter(a for a, _ in pairs), Counter(b for _, b in pairs)
    pe = sum(ca[c] * cb[c] for c in set(ca) | set(cb)) / (n * n)
    return None if pe >= 1 else (po - pe) / (1 - pe)


def jaccard(a, b) -> float:
    a, b = set(a or []), set(b or [])
    return 1.0 if not a and not b else len(a & b) / len(a | b)


def _number(s):
    t = str(s).strip().replace(",", "").replace("−", "-").replace("%", "").replace("+", "").strip()
    m = re.fullmatch(r"-?\d+(\.\d+)?", t)
    if not m:
        return None, None
    return float(t), len(m.group(1)) - 1 if m.group(1) else 0


def values_agree(first, second) -> bool:
    """Exact match within rounding: both printed numbers rounded to the coarser of their two precisions."""
    a, da = _number(first)
    b, db = _number(second)
    if a is None or b is None:
        return re.sub(r"\s+", "", str(first)) == re.sub(r"\s+", "", str(second))
    d = min(da, db)
    return round(a, d) == round(b, d)


def agreement(first: dict, second: dict, matches: dict) -> tuple[list[dict], list[dict], list[dict]]:
    """Agreement between the first coding (record files) and the second pass, before any disagreement is resolved.
    first: report id -> record; second: report id -> second-pass document; matches: report id -> list (one entry per
    second-pass defence, in order) of first-pass defence ids. Returns (field rows, per-report rows, disagreements)."""
    cat = defaultdict(list)          # field -> [(first, second)]
    multi = defaultdict(list)        # field -> [jaccard]
    multi_exact = defaultdict(list)
    vals, set_j, dis, per_report = [], [], [], []
    for rid in sorted(second):
        rec, sp, mt = first[rid], second[rid], matches[rid]
        cat["role"].append((rec["report"]["role"], sp["role"]))
        if rec["report"]["role"] != sp["role"]:
            dis.append({"report": rid, "unit": "report", "field": "role", "first": rec["report"]["role"], "second": sp["role"]})
        for q in QUANTITIES:
            a, b = rec["coverage"].get(q), sp["coverage"].get(q)
            cat[f"coverage.{q}"].append((a, b))
            if a != b:
                dis.append({"report": rid, "unit": "coverage", "field": q, "first": a, "second": b})
        dfs = {d["id"]: d for d in rec.get("defences") or []}
        if len(mt) != len(sp["defences"]):
            raise ValueError(f"{rid}: {len(mt)} match lines for {len(sp['defences'])} second-pass defences")
        matched = set()
        for entry, ids in zip(sp["defences"], mt):
            for did in ids or []:
                if did not in dfs:
                    raise ValueError(f"{rid}: match {did!r} is not a first-pass defence")
                matched.add(did)
                d = dfs[did]
                for f in DEF_CATEGORICAL:
                    cat[f"defence.{f}"].append((d.get(f), entry.get(f)))
                    if d.get(f) != entry.get(f):
                        dis.append({"report": rid, "unit": did, "field": f, "first": d.get(f), "second": entry.get(f)})
                for f in DEF_MULTI:
                    j = jaccard(d.get(f), entry.get(f))
                    multi[f"defence.{f}"].append(j)
                    multi_exact[f"defence.{f}"].append(set(d.get(f) or []) == set(entry.get(f) or []))
                    if j < 1:
                        dis.append({"report": rid, "unit": did, "field": f, "first": ";".join(d.get(f) or []),
                                    "second": ";".join(entry.get(f) or [])})
        # defence set at the method level: second-pass entries against first-pass groups; first-pass defences no
        # entry matched are grouped by name (with the base model removed) so one missed method counts once
        unmatched_first = [d for i, d in dfs.items() if i not in matched]
        groups = {re.sub(r"\s+on\s+.*$", "", d["name"]).strip().lower() for d in unmatched_first}
        n_entries = len(sp["defences"])
        n_matched_entries = sum(1 for ids in mt if ids)
        j = n_matched_entries / (n_entries + len(groups)) if (n_entries + len(groups)) else 1.0
        set_j.append(j)
        for d in unmatched_first:
            dis.append({"report": rid, "unit": d["id"], "field": "defence set", "first": d["name"], "second": "(not coded)"})
        for entry, ids in zip(sp["defences"], mt):
            if not ids:
                dis.append({"report": rid, "unit": "second pass", "field": "defence set", "first": "(not coded)", "second": entry["name"]})
        res = {r["id"]: r for r in rec.get("results") or []}
        n_ok = 0
        for v in sp.get("values") or []:
            r = res.get(v["result"])
            if r is None:
                raise ValueError(f"{rid}: value card {v['result']!r} is not a first-pass result")
            ok = values_agree(r["value"], v["value"])
            n_ok += ok
            vals.append(ok)
            if not ok:
                dis.append({"report": rid, "unit": v["result"], "field": "value", "first": r["value"], "second": v["value"]})
        per_report.append({"report": rid, "first_defences": len(dfs), "second_entries": n_entries,
                           "matched_entries": n_matched_entries, "unmatched_first_groups": len(groups),
                           "defence_set_jaccard": round(j, 3), "values": len(sp.get("values") or []), "values_agree": n_ok})
    rows = []

    def add(ftype, field, n, exact=None, kappa=None, jac=None):
        ok = True
        if exact is not None and exact < EXACT_MIN:
            ok = False
        if kappa is not None and kappa < KAPPA_MIN:
            ok = False
        if jac is not None and jac < JACCARD_MIN:
            ok = False
        rows.append({"field_type": ftype, "field": field, "pairs": n,
                     "exact_agreement": "" if exact is None else round(exact, 3),
                     "kappa": "" if kappa is None else round(kappa, 3),
                     "mean_jaccard": "" if jac is None else round(jac, 3),
                     "meets_threshold": "yes" if ok else "no", "action": "none" if ok else "re-code this field in full"})
    for f, pairs in cat.items():
        add("categorical", f, len(pairs), sum(a == b for a, b in pairs) / len(pairs), cohen_kappa(pairs))
    allc = [p for ps in cat.values() for p in ps]
    add("categorical (pooled)", "all categorical fields", len(allc), sum(a == b for a, b in allc) / len(allc), cohen_kappa(allc))
    rows[-1]["action"] = "summary row (the rule applies field by field)"
    for f, js in multi.items():
        add("multi-label", f, len(js), sum(multi_exact[f]) / len(js), None, sum(js) / len(js))
    add("multi-label", "defence set (method level, per report)", len(set_j), None, None, sum(set_j) / len(set_j))
    add("numeric", "printed values re-read from their addresses", len(vals), sum(vals) / len(vals) if vals else 0)
    return rows, per_report, dis


# ---------------------------------------------------------------- 04r: applying a listed re-code, and the amended rules as checks
CHANGE_FIELDS = ["report", "unit", "field", "old", "new", "reason", "page"]
LIST_FIELDS = {"axis1", "axis2", "also_quantity"}
PROMPT_OR_RULE = {"prompting", "fixed_inference_rule"}


def encode(v) -> str:
    """One cell for a field value: lists joined by ';', absent as ''."""
    if v is None:
        return ""
    return ";".join(str(x) for x in v) if isinstance(v, list) else str(v)


def _decode(field: str, s: str):
    if s == "":
        return None
    if field.split(".")[-1] in LIST_FIELDS:
        return s.split(";")
    if field.startswith("pages."):
        return int(s)
    return s


def _target(rec: dict, unit: str) -> dict:
    if unit in ("report", "coverage"):
        return rec[unit]
    kind, _, rid = unit.partition(":")
    for row in rec.get(kind) or []:
        if row.get("id") == rid:
            return row
    raise KeyError(unit)


def field_value(rec: dict, unit: str, field: str):
    obj = _target(rec, unit)
    if "." in field:
        a, b = field.split(".", 1)
        return (obj.get(a) or {}).get(b)
    return obj.get(field)


def apply_changes(records: dict, rows: list[dict]) -> list[str]:
    """Applies listed changes in place. A row whose old value is not the value on file is refused, so a change list
    built from other files than the committed ones cannot be applied by mistake."""
    problems = []
    for r in rows:
        rec = records.get(r["report"])
        if rec is None:
            problems.append(f"{r['report']}: no such record")
            continue
        try:
            obj = _target(rec, r["unit"])
        except KeyError:
            problems.append(f"{r['report']}:{r['unit']}: no such unit")
            continue
        cur = encode(field_value(rec, r["unit"], r["field"]))
        if cur != r["old"]:
            problems.append(f"{r['report']}:{r['unit']}:{r['field']} is {cur!r} on file, the change list expects {r['old']!r}")
            continue
        new = _decode(r["field"], r["new"])
        if "." in r["field"]:
            a, b = r["field"].split(".", 1)
            sub = obj.get(a) or {}
            if new is None:
                sub.pop(b, None)
            else:
                sub[b] = new
            if sub:
                obj[a] = sub
            else:
                obj.pop(a, None)
        elif new is None:
            obj.pop(r["field"], None)
        else:
            obj[r["field"]] = new
    return problems


def rule_problems(rec: dict) -> list[str]:
    """Amended rules 8, 9 and 32 (coding_rules.md, Amendment 2) checked on one record."""
    out = []
    has_none = any(str(e.get("defence_id", "")).startswith("none:") for e in rec.get("evaluations") or [])
    for d in rec.get("defences") or []:
        a1, sig = d["axis1"], d["signal_dependence"]
        trained = not set(d["axis2"]) <= PROMPT_OR_RULE
        tag = f"defences:{d['id']}"
        if "no_signal" in a1 and len(a1) > 1:
            out.append(f"{tag}: no_signal combined with another axis 1 value")
        if (a1 == ["no_signal"]) != (sig == "not_applicable"):
            out.append(f"{tag}: axis 1 {encode(a1)} with signal_dependence {sig} (not_applicable exactly when axis 1 is no_signal)")
        if trained and sig == "baseline_only":
            out.append(f"{tag}: baseline_only on a trained defence")
        if not trained and sig == "stated":
            out.append(f"{tag}: stated on a prompting defence or fixed rule (measured, baseline_only or not_reported)")
        if sig == "baseline_only" and not has_none:
            out.append(f"{tag}: baseline_only but the report has no undefended evaluation")
        if sig == "measured_other_checkpoint" and not d.get("signal_dependence_detail"):
            out.append(f"{tag}: measured_other_checkpoint without signal_dependence_detail naming the checkpoint")
        if d["origin"] == "other_report" and d["isolation"] == "no_ablation":
            out.append(f"{tag}: no_ablation on another report's defence (not_applicable unless this report ablates it)")
    cov = rec.get("coverage") or {}
    for q in ("R1", "R3ns", "R3cp", "R3safe"):
        rs = [x for x in rec.get("results") or [] if x.get("quantity") == q or q in (x.get("also_quantity") or [])]
        if cov.get(q) in ("direct", "proxy") and not rs:
            out.append(f"coverage {q} is {cov.get(q)} but no result carries {q}")
        if cov.get(q) in ("not_reported", "not_applicable") and rs:
            out.append(f"coverage {q} is {cov.get(q)} but {len(rs)} results carry {q}")
        main = [x["coverage"] for x in rs if x.get("quantity") == q]
        if main and len(main) == len(rs) and cov.get(q) in ("direct", "proxy"):
            best = "direct" if "direct" in main else "proxy"
            if best != cov.get(q):
                out.append(f"coverage {q} is {cov.get(q)} but the best result coded for it is {best}")
    return out


# ---------------------------------------------------------------- 04_full_extraction: decisive values and the map
AXIS1 = ["chat_roles", "delimiters_markers", "encodings", "learned_token_features", "fixed_embedding_transform", "position",
         "internal_intervention", "no_signal"]
AXIS2 = ["prompting", "soft_tokens_frozen", "sft", "preference_optimisation", "distillation", "adversarial_rl", "fixed_inference_rule"]


def value_count(value: str, page_text: str, norm) -> int:
    """How many times a printed value stands on the page as a number of its own; norm is schema.norm_value."""
    v = norm(value).replace(" ", "")
    if not v:
        return 0
    core = re.escape(v[:-1] if v.endswith("%") else v)
    return len(re.findall(rf"(?<![\d.]){core}(?![\d])", norm(page_text or "")))


def decisive_results(rec: dict, grade) -> dict:
    """Plan v2 A6: the printed values in a grade-A comparison. Returns {result id: set of reasons}; the reasons say
    whether the comparison supports or the result contradicts a headline claim."""
    by_eval = defaultdict(list)
    for r in rec.get("results") or []:
        by_eval[r["evaluation_id"]].append(r)
    comps = {c["id"]: c for c in rec.get("comparisons") or []}
    support = {cid for k in rec.get("claims") or [] for cid in k.get("supported_by") or []}
    contra = {x for k in rec.get("claims") or [] for x in k.get("contradicted_by") or []}
    out = defaultdict(set)
    for cid, c in comps.items():
        if grade(c) != "A":
            continue
        tag = "grade-A comparison behind a claim" if cid in support else "grade-A comparison"
        for side in ("left", "right"):
            for r in by_eval.get(c[side], []):
                out[r["id"]].add(tag)
                if cid in contra:
                    out[r["id"]].add("contradicts a claim")
    for r in rec.get("results") or []:
        if r["id"] in contra and r["id"] in out:
            out[r["id"]].add("contradicts a claim")
    return {k: v for k, v in out.items()
            if next(r for r in rec["results"] if r["id"] == k).get("value_source") != "figure_only"}


def map_cells(records: dict) -> list[dict]:
    """Defence records per cell of the map (axis 1 by axis 2), from reports read in full. A defence with two axis-1 or
    two axis-2 values counts in every cell it occupies."""
    cell = defaultdict(lambda: {"records_this_report": 0, "records_all": 0, "reports": set()})
    for rid, rec in records.items():
        if rec["report"].get("read") != "F":
            continue
        for d in rec.get("defences") or []:
            for a in d["axis1"]:
                for b in d["axis2"]:
                    c = cell[(a, b)]
                    c["records_all"] += 1
                    if d["origin"] == "this_report":
                        c["records_this_report"] += 1
                        c["reports"].add(rid)
    rows = []
    for a in AXIS1:
        for b in AXIS2:
            c = cell.get((a, b), {"records_this_report": 0, "records_all": 0, "reports": set()})
            rows.append({"axis1": a, "axis2": b, "defence_records_this_report": c["records_this_report"],
                         "defence_records_all": c["records_all"], "reports_proposing": len(c["reports"]),
                         "report_ids": ";".join(sorted(c["reports"]))})
    return rows

In [ ]:
%%writefile tests/test_extract.py
"""Tests for src/extract.py: the seeded sample, value cards, tables, cross-record checks and agreement statistics."""
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))
extract = importlib.import_module("extract")
schema_mod = importlib.import_module("schema")


def _rec(rid="r_a", slug="s_a"):
    return {"report": {"id": rid, "slug": slug, "role": "proposes", "read": "F"},
            "coverage": {q: "not_reported" for q in extract.QUANTITIES} | {"R1": "direct"},
            "defences": [{"id": "d1", "name": "Def", "origin": "this_report", "base_checkpoint": "M", "axis1": ["chat_roles"],
                          "axis2": ["sft"], "isolation": "no_ablation", "signal_dependence": "not_reported",
                          "trains_instruction_pair": "no", "trains_carry_through": "no", "page": 2}],
            "evaluations": [{"id": "e1", "defence_id": "d1", "benchmark": "B", "attacker_type": "static", "attacker_access": "black_box",
                             "judge": "J", "layout_same_as_defence": "yes", "measured_by": rid, "page": 3},
                            {"id": "e0", "defence_id": "none:M", "benchmark": "B", "attacker_type": "static", "attacker_access": "black_box",
                             "judge": "J", "layout_same_as_defence": "yes", "measured_by": rid, "page": 3}],
            "results": [{"id": f"x{i}", "evaluation_id": "e1" if i % 2 else "e0", "quantity": "R1", "coverage": "direct",
                         "metric_as_printed": "ASR", "value": f"{i}.5", "unit": "percent", "direction": "lower_better",
                         "qualifier": "exact", "value_source": "table", "page": 3} for i in range(9)],
            "comparisons": [{"id": "c1", "left": "e1", "right": "e0", "purpose": "W", "varies": "defence",
                             **{c: "matched" for c in schema_mod.CONDITIONS}, "page": 3}],
            "claims": [{"id": "k1", "text": "t", "location": "abstract", "anchor": "a", "jobs": ["R1"], "supported_by": ["c1"], "page": 1}]}


def test_sample_is_stable_and_twenty_percent():
    slugs = [f"s{i:03d}" for i in range(124)]
    s = extract.second_pass_sample(slugs)
    assert len(s) == 25 and len(set(s)) == 25
    dropped = [x for x in slugs if x != slugs[-1] and x not in s[:1]]
    s2 = extract.second_pass_sample(dropped)
    assert set(s[1:]) <= set(s2)                       # removing one study does not reorder the others
    assert extract.second_pass_sample(slugs) == s      # deterministic


def test_cards_hide_values_and_are_capped():
    cards = extract.value_cards(_rec())
    assert len(cards) == extract.CARDS_PER_REPORT
    assert all("value" not in c for c in cards)
    assert {c["arm"].startswith(("Def on", "no defence")) for c in cards} == {True}


def test_tables_have_global_ids_and_grades():
    sch = schema_mod.load_schema(Path(__file__).resolve().parent.parent / "schema" / "record_schema.yaml")
    t = extract.build_tables({"r_a": _rec()}, sch, schema_mod.grade)
    assert t["defences"][0][0]["defence_uid"] == "r_a:d1"
    ev = {r["evaluation_uid"]: r for r in t["evaluations"][0]}
    assert ev["r_a:e1"]["measurement"] == "original" and ev["r_a:e0"]["measurement"] == "undefended"
    assert t["comparisons"][0][0]["grade"] == "A"
    assert t["claims"][0][0]["supported_by"] == "r_a:c1"


def test_cross_record_checks():
    recs = {"r_a": _rec()}
    assert extract.cross_record_problems(recs, ["s_a"]) == []
    probs = extract.cross_record_problems(recs, ["s_a", "s_b"], {"r_a": 2})
    assert any("s_b has no record" in p for p in probs)
    assert any("beyond the PDF's 2 pages" in p for p in probs)


def test_kappa_jaccard_and_rounding():
    assert extract.cohen_kappa([("a", "a"), ("b", "b")]) == 1.0
    assert extract.cohen_kappa([("a", "a"), ("a", "a")]) is None
    assert abs(extract.cohen_kappa([("a", "a"), ("a", "b"), ("b", "b"), ("b", "a")])) < 1e-9
    assert extract.jaccard(["x", "y"], ["x"]) == 0.5
    assert extract.values_agree("45.20%", "45.2") and extract.values_agree("+0.028", "0.028")
    assert not extract.values_agree("45.2", "45.3") and extract.values_agree("12", "12.0")


def test_agreement_counts_disagreements():
    rec = _rec()
    sp = {"report": "r_a", "role": "proposes", "coverage": dict(rec["coverage"]),
          "defences": [{"name": "Def", "bases": ["M"], "origin": "this_report", "axis1": ["chat_roles", "position"], "axis2": ["sft"],
                        "isolation": "ablation_present", "signal_dependence": "not_reported", "trains_instruction_pair": "no",
                        "trains_carry_through": "no"}],
          "values": [{"result": "x1", "value": "1.5"}, {"result": "x2", "value": "9"}]}
    rows, per, dis = extract.agreement({"r_a": rec}, {"r_a": sp}, {"r_a": [["d1"]]})
    by = {r["field"]: r for r in rows}
    assert by["defence.isolation"]["exact_agreement"] == 0.0
    assert by["defence.axis1"]["mean_jaccard"] == 0.5
    assert by["printed values re-read from their addresses"]["exact_agreement"] == 0.5
    assert {d["field"] for d in dis} == {"isolation", "axis1", "value"}
    assert extract.validate_second_pass(sp, schema_mod.load_schema(Path(__file__).resolve().parent.parent / "schema" / "record_schema.yaml"),
                                        [{"result": "x1"}, {"result": "x2"}], rec, [["d1"]]) == []


def test_apply_changes_checks_the_old_value_and_edits_in_place():
    recs = {"r_a": _rec()}
    rows = [{"report": "r_a", "unit": "defences:d1", "field": "axis1", "old": "chat_roles", "new": "no_signal", "reason": "", "page": ""},
            {"report": "r_a", "unit": "defences:d1", "field": "pages.isolation", "old": "", "new": "5", "reason": "", "page": "5"},
            {"report": "r_a", "unit": "results:x1", "field": "also_quantity", "old": "", "new": "R3cp", "reason": "", "page": ""},
            {"report": "r_a", "unit": "coverage", "field": "R3cp", "old": "not_reported", "new": "proxy", "reason": "", "page": ""},
            {"report": "r_a", "unit": "coverage", "field": "R1", "old": "proxy", "new": "direct", "reason": "", "page": ""}]
    probs = extract.apply_changes(recs, rows)
    d = recs["r_a"]["defences"][0]
    assert d["axis1"] == ["no_signal"] and d["pages"] == {"isolation": 5}
    assert recs["r_a"]["results"][1]["also_quantity"] == ["R3cp"] and recs["r_a"]["coverage"]["R3cp"] == "proxy"
    assert len(probs) == 1 and "expects 'proxy'" in probs[0]          # R1 is direct on file: the row is refused
    back = [{"report": "r_a", "unit": "defences:d1", "field": "pages.isolation", "old": "5", "new": "", "reason": "", "page": ""}]
    assert extract.apply_changes(recs, back) == [] and "pages" not in d


def test_rule_problems_follow_amendment_2():
    rec = _rec()
    assert extract.rule_problems(rec) == []
    d = rec["defences"][0]
    d["axis1"], d["signal_dependence"] = ["no_signal"], "baseline_only"
    probs = extract.rule_problems(rec)
    assert any("not_applicable exactly when" in p for p in probs) and any("baseline_only on a trained" in p for p in probs)
    d["axis1"], d["axis2"], d["signal_dependence"] = ["delimiters_markers"], ["prompting"], "stated"
    assert any("stated on a prompting" in p for p in extract.rule_problems(rec))
    d["signal_dependence"], d["origin"], d["isolation"] = "baseline_only", "other_report", "no_ablation"
    assert any("another report's defence" in p for p in extract.rule_problems(rec))
    d["origin"] = "this_report"
    rec["coverage"]["R3cp"] = "proxy"
    assert any("coverage R3cp is proxy but no result" in p for p in extract.rule_problems(rec))


def test_decisive_values_and_map_cells():
    rec = _rec()
    dec = extract.decisive_results(rec, schema_mod.grade)
    assert set(dec) == {f"x{i}" for i in range(9)}                   # every printed value of both arms of the grade-A comparison
    assert all("grade-A comparison behind a claim" in v for v in dec.values())
    rec["comparisons"][0]["c4_attacker"] = "unmatched"                # grade B: nothing is decisive
    assert extract.decisive_results(rec, schema_mod.grade) == {}
    assert extract.value_count("1.5", "a 1.5 b 11.5 c 1.5%", schema_mod.norm_value) == 2
    cells = {(r["axis1"], r["axis2"]): r for r in extract.map_cells({"r_a": _rec()})}
    assert len(cells) == len(extract.AXIS1) * len(extract.AXIS2)
    assert cells[("chat_roles", "sft")]["defence_records_this_report"] == 1 and cells[("no_signal", "sft")]["defence_records_all"] == 0

In [ ]:
import schema, search, corpus, paths, extract
for m in (schema, search, corpus, paths, extract):
    importlib.reload(m)
t = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_extract.py"], capture_output=True, text=True)
print(t.stdout.strip().splitlines()[-1] if t.stdout.strip() else t.stderr.strip()[-400:])

## Stage 3. Every record against the frozen schema, its PDF, the amended rules and the corpus

In [ ]:
import re, logging, datetime as dt
from collections import Counter
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader
import pypdfium2 as pdfium

S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
included = sorted(s for s, m in man.items() if m.get("decision") == "include")
recs = {f.stem: schema.load_record(f) for f in sorted(paths.RECORDS.glob("*.yaml"))}
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()

texts, pages, problems, check_rows, summary = {}, {}, [], [], []
for i, rid in enumerate(sorted(recs)):
    rec = recs[rid]
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    probs, notes, third, third_chart = [], [], [], []
    if not pdf.exists():
        probs.append(f"{pdf.name} not in data/defences")
    else:
        if corpus.sha256(pdf) != man.get(slug, {}).get("sha256"):
            probs.append("PDF differs from the one registered in reports/defences.csv")
        rd = PdfReader(str(pdf))
        plain = [p.extract_text() or "" for p in rd.pages]
        lay = [p.extract_text(extraction_mode="layout") or "" for p in rd.pages]
        texts[slug], pages[rid] = (plain, lay), len(rd.pages)
        p0, notes = schema.validate_record(rec, S1, plain, lay)
        for p in p0:                                   # a value pypdf glues to its neighbour is looked for in a third reading
            m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
            if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
                third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
            else:
                probs.append(p)
        for r in rec.get("results") or []:            # chart values kept as printed are found on their page too
            if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
                pg = int(r["page"]) - 1
                if not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                    if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                        third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                    else:
                        probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):
        dup = [x for x, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {x} used more than once" for x in dup]
    for e in rec.get("evaluations") or []:
        if not e.get("measured_by"):
            probs.append(f"evaluations:{e['id']}: measured_by not filled")
    probs += extract.rule_problems(rec)
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_chart = sum(1 for r in rec.get("results") or [] if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""))
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"report": rid, "slug": slug, "pdf_pages": pages.get(rid, ""), "defences": len(rec.get("defences") or []),
                    "evaluations": len(rec.get("evaluations") or []), "results": len(rec.get("results") or []),
                    "values_checked": n_val, "values_third_reading": len(third), "chart_values_checked": n_chart,
                    "chart_values_third_reading": len(third_chart), "anchors_checked": n_anc,
                    "comparisons": len(rec.get("comparisons") or []), "claims": len(rec.get("claims") or []),
                    "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    check_rows += [{"report": rid, "status": "value found in the third reading", "detail": t} for t in third + third_chart]
    check_rows += [{"report": rid, "status": "residual note", "detail": n} for n in notes]
    problems += [f"{rid}: {p}" for p in probs]
    if (i + 1) % 20 == 0:
        print(f"  {i + 1} of {len(recs)} records checked")
problems += extract.cross_record_problems(recs, included, pages)
for p in problems:
    print("PROBLEM:", p)
out_dir = paths.REPORTS / "extraction"
out_dir.mkdir(parents=True, exist_ok=True)
search.write_csv(out_dir / "check_04.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(out_dir / "summary_04.csv", summary, list(summary[0].keys()))
tot = {k: sum(s[k] for s in summary) for k in summary[0] if k not in ("report", "slug", "pdf_pages")}
print(f"{len(recs)} records, {len(included)} included studies; values {tot['values_checked']} "
      f"({tot['values_third_reading']} found only in the third reading), chart values {tot['chart_values_checked']}, "
      f"anchors {tot['anchors_checked']}; problems {len(problems)}")
if problems:
    raise SystemExit("STOP: the records do not validate; see reports/extraction/check_04.csv. No table was written.")

## Stage 4. The extraction tables, one row per record with global ids

In [ ]:
tables = extract.build_tables(recs, S1, schema.grade)
for name, (rows, fields) in tables.items():
    search.write_csv(out_dir / f"{name}.csv", rows, fields)
    print(f"reports/extraction/{name}.csv: {len(rows)} rows, {len(fields)} columns")
grades = Counter(r["grade"] for r in tables["comparisons"][0])
meas = Counter(r["measurement"] for r in tables["evaluations"][0])
print("comparison grades:", dict(sorted(grades.items())), "| evaluations:", dict(meas))

## Stage 5. Decisive values: every printed value in a grade-A comparison, with its address and the code's check

In [ ]:
dec_rows = []
for rid in sorted(recs):
    rec = recs[rid]
    slug = rec["report"]["slug"]
    plain, lay = texts[slug]
    evs = {e["id"]: e for e in rec.get("evaluations") or []}
    names = {d["id"]: d["name"] for d in rec.get("defences") or []}
    for xid, why in sorted(extract.decisive_results(rec, schema.grade).items()):
        r = next(x for x in rec["results"] if x["id"] == xid)
        e = evs.get(r["evaluation_id"], {})
        pg = int(r["page"])
        n_plain = extract.value_count(r["value"], plain[pg - 1], schema.norm_value)
        n_lay = extract.value_count(r["value"], lay[pg - 1], schema.norm_value)
        if n_plain or n_lay:
            found, n = "pypdf", max(n_plain, n_lay)
        else:
            found, n = "pypdfium2 (third reading)", extract.value_count(r["value"], pdfium_page(slug, pg), schema.norm_value)
        if n == 0:
            raise SystemExit(f"STOP: decisive value {rid}:{xid} not found on page {pg}")
        check = ("one occurrence on the page" if n == 1 and found == "pypdf" else
                 "more than one occurrence on the page: read the cell" if found == "pypdf" else
                 "third reading only: read the cell")
        did = e.get("defence_id", "")
        dec_rows.append({"result_uid": f"{rid}:{xid}", "report": rid, "page": pg, "location": r.get("location", ""),
                         "arm": "undefended (" + did[5:] + ")" if str(did).startswith("none:") else names.get(did, did),
                         "benchmark": e.get("benchmark", ""), "quantity": r["quantity"], "metric_as_printed": r["metric_as_printed"],
                         "value": r["value"], "unit": r.get("unit", ""), "value_source": r.get("value_source", ""),
                         "why_decisive": "; ".join(sorted(why)), "found_by": found, "occurrences_on_page": n, "code_check": check})
search.write_csv(out_dir / "decisive_values_04.csv", dec_rows, list(dec_rows[0].keys()))
cc = Counter(r["code_check"] for r in dec_rows)
claim_linked = sum("behind a claim" in r["why_decisive"] or "contradicts" in r["why_decisive"] for r in dec_rows)
print(f"decisive values: {len(dec_rows)} in {len({r['report'] for r in dec_rows})} reports; behind or against a headline claim: {claim_linked}")
for k, v in cc.most_common():
    print(f"  {k}: {v}")
for d in _docs.values():                               # close the third-reading documents
    d.close()
_docs.clear()

## Stage 6. Gate G3, decision record and tests

In [ ]:
cells = extract.map_cells(recs)
search.write_csv(out_dir / "map_cells_04.csv", cells, list(cells[0].keys()))
populated = [c for c in cells if c["defence_records_all"]]
proposed = [c for c in cells if c["defence_records_this_report"]]
all_f = all(r["report"].get("read") == "F" for r in recs.values())
comp_rows = tables["comparisons"][0]
stated = sum(1 for c in comp_rows if c.get("c2_layout") != "unreported")
share_stated = stated / len(comp_rows)
sp = Path("reports/decision_04sp.txt").read_text().splitlines()
below = next((l.split(": ", 1)[1] for l in sp if l.startswith("below the threshold")), "")
g3_rows = all_f and all(c["defence_records_all"] for c in populated)
g3_layout = share_stated > 0.5
g3_agree = "AMENDED RULES HOLD" in d04r[0]
g3 = "PASSED" if (g3_rows and g3_layout and g3_agree) else "NOT PASSED"
lines = [f"decision_04: FULL EXTRACTION CHECKED; GATE G3 {g3}",
         f"this step: {dt.date.today().isoformat()}",
         f"corpus: {len(included)} included studies, {len(recs)} records, every report read in full: {all_f}",
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, {tot['comparisons']} comparisons, "
         f"{tot['claims']} claims",
         f"checked against the PDFs in one run: {tot['values_checked']} printed values ({tot['values_third_reading']} found only in the third "
         f"reading), {tot['chart_values_checked']} chart values ({tot['chart_values_third_reading']} found only in the third reading), "
         f"{tot['anchors_checked']} anchors; every PDF matches its registered SHA-256; amended rules hold; 0 problems",
         "tables: " + ", ".join(f"reports/extraction/{n}.csv ({len(t[0])})" for n, t in tables.items()),
         "comparison grades: " + ", ".join(f"{g} {grades[g]}" for g in "ABCU") +
         "; evaluations: " + ", ".join(f"{k} {v}" for k, v in sorted(meas.items())),
         f"decisive values (printed values in grade-A comparisons): {len(dec_rows)}; behind or against a headline claim: {claim_linked}; "
         + "; ".join(f"{k} {v}" for k, v in cc.most_common()),
         "decisive values still to read against the page: the values the findings of notebook 05 use, wherever the code could not pin "
         "the value to one place on its page",
         f"map (full-text rows per cell, reports/extraction/map_cells_04.csv): {len(populated)} of {len(cells)} cells hold defence records, "
         f"{len(proposed)} hold a defence proposed or defined by its report; the empty cells are a finding of the map, not missing reading",
         *[f"  {c['axis1']} x {c['axis2']}: {c['defence_records_this_report']} records (reports: {c['reports_proposing']}; "
           f"with other reports' defences: {c['defence_records_all']})" for c in proposed],
         f"G3 full-text rows: {'met' if g3_rows else 'NOT met'} (every populated cell has rows from reports read in full)",
         f"G3 layouts stated: {'met' if g3_layout else 'NOT met'} ({stated} of {len(comp_rows)} comparisons, {share_stated:.0%}, state the layout condition)",
         f"G3 agreement: below the threshold before resolution on {below or 'no field'}; re-coded in full in notebook 04r ({R_COMMIT}), "
         "as the gate prescribes",
         "next: notebook 05_analysis (map counts, metric crosswalk, coverage by paper, by configuration and jointly, grades and claim support, "
         "stress-test coverage, signal dependence, layout compatibility, suppression cost from published pairs)"]
(paths.REPORTS / "decision_04.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 7. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04 full extraction: all 124 records checked against their PDFs in one run, extraction tables, decisive values listed, gate G3"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)